# Unequal-Axis Grid Spacing Verification

This notebook verifies rectilinear grids with constant but unequal `dx`,
`dy`, and `dz`. It checks the Python contract, CFL limit, axis-specific
field updates, source scaling, CPML coefficients, 2D and 3D forward and
backward propagation, material gradients, and optional CPU/CUDA parity.

Spatially varying cell sizes within a single axis are outside the scope of
this implementation and this verification.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


Repository root: /Users/llsra/Desktop/DeepGPR
DeepGPR package: /Users/llsra/Desktop/DeepGPR/src/DeepGPR/__init__.py


In [2]:
import math
import torch

from DeepGPR.common import (
    _normalize_grid_spacing,
    buildpmlcoeffs,
    check_cfl,
)

torch.manual_seed(2026)
CPU = torch.device("cpu")
CUDA = vu.selected_cuda_device()
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, CPU)
SPACING = (0.02, 0.015, 0.01)


In [3]:
normalized_inputs = [
    _normalize_grid_spacing(SPACING),
    _normalize_grid_spacing(list(SPACING)),
    _normalize_grid_spacing(torch.tensor(SPACING, dtype=torch.float64)),
]
vu.record_check(
    CHECKS,
    "three-value spacing accepts tuple, list, and tensor inputs",
    all(value == SPACING for value in normalized_inputs),
    normalized_inputs=normalized_inputs,
)

nx, ny, nt = 14, 18, 100
er_equal = torch.full((nx, ny), 4.0)
se_equal = torch.zeros_like(er_equal)
source_equal = DeepGPR.wavelet.ricker(3.0e8, nt, 3.0e-11, 3.0e-9).reshape(1, nt, 1)
source_location_equal = torch.tensor([[[6, 7, 0]]], dtype=torch.int32)
receiver_location_equal = torch.tensor([[[6, 11, 0]]], dtype=torch.int32)
equal_arguments = dict(
    device=CPU,
    dt=3.0e-11,
    source_amplitudes=source_equal,
    source_location=source_location_equal,
    receiver_location=receiver_location_equal,
    er=er_equal,
    se=se_equal,
    pmlthick=3,
    fdtd_order=2,
    mode=2,
)
scalar_receiver = DeepGPR.compute(dx=0.02, **equal_arguments)[-1]
sequence_receiver = DeepGPR.compute(dx=[0.02, 0.02, 0.02], **equal_arguments)[-1]
tensor_receiver = DeepGPR.compute(
    dx=torch.tensor([0.02, 0.02, 0.02], dtype=torch.float64),
    **equal_arguments,
)[-1]
sequence_error = vu.max_abs_difference(sequence_receiver, scalar_receiver)
tensor_error = vu.max_abs_difference(tensor_receiver, scalar_receiver)
vu.record_check(
    CHECKS,
    "scalar spacing preserves exact equal-axis behavior",
    sequence_error == 0.0 and tensor_error == 0.0,
    sequence_max_abs_difference=sequence_error,
    tensor_max_abs_difference=tensor_error,
)


[PASS] three-value spacing accepts tuple, list, and tensor inputs
{
  "normalized_inputs": [
    [
      0.02,
      0.015,
      0.01
    ],
    [
      0.02,
      0.015,
      0.01
    ],
    [
      0.02,
      0.015,
      0.01
    ]
  ]
}
[PASS] scalar spacing preserves exact equal-axis behavior
{
  "sequence_max_abs_difference": 0.0,
  "tensor_max_abs_difference": 0.0
}


In [4]:
cfl_rows = []
for shape in ((12, 14, 1), (10, 12, 14)):
    active_spacing = [
        spacing for size, spacing in zip(shape, SPACING) if size > 1
    ]
    dt_limit = math.sqrt(4.0) / (
        vu.C0 * math.sqrt(sum(1.0 / spacing**2 for spacing in active_spacing))
    )
    er_cfl = torch.full(shape, 4.0)
    mr_cfl = torch.ones_like(er_cfl)
    check_cfl(
        SPACING,
        0.99 * dt_limit,
        *shape,
        er=er_cfl,
        mr=mr_cfl,
        fdtd_order=2,
    )
    rejected_above_limit = False
    try:
        check_cfl(
            SPACING,
            1.01 * dt_limit,
            *shape,
            er=er_cfl,
            mr=mr_cfl,
            fdtd_order=2,
        )
    except ValueError as exc:
        rejected_above_limit = "CFL" in str(exc)
    row = {
        "shape": shape,
        "independent_dt_limit": dt_limit,
        "accepted_dt": 0.99 * dt_limit,
        "rejected_dt": 1.01 * dt_limit,
        "rejected_above_limit": rejected_above_limit,
    }
    cfl_rows.append(row)
    vu.record_check(
        CHECKS,
        f"{len(active_spacing)}D CFL boundary uses every active axis spacing",
        rejected_above_limit,
        **row,
    )


[PASS] 2D CFL boundary uses every active axis spacing
{
  "accepted_dt": 7.925482901908092e-11,
  "independent_dt_limit": 8.005538284755649e-11,
  "rejected_above_limit": true,
  "rejected_dt": 8.085593667603206e-11,
  "shape": [
    12,
    14,
    1
  ]
}
[PASS] 3D CFL boundary uses every active axis spacing
{
  "accepted_dt": 5.073770513609131e-11,
  "independent_dt_limit": 5.125020720817304e-11,
  "rejected_above_limit": true,
  "rejected_dt": 5.176270928025477e-11,
  "shape": [
    10,
    12,
    14
  ]
}


In [5]:
def updated_ez_from_x(dx_value):
    nx, ny = 8, 10
    shape = (1, nx + 1, ny + 1, 2)
    electric = tuple(torch.zeros(shape) for _ in range(3))
    hy = torch.zeros(shape)
    hy[:] = torch.arange(nx + 1, dtype=torch.float32).reshape(1, nx + 1, 1, 1)
    magnetic = (torch.zeros(shape), hy, torch.zeros(shape))
    model = torch.full((nx, ny), 4.0)
    location = torch.tensor([[[4, 5, 0]]], dtype=torch.int32)
    return DeepGPR.compute(
        device=CPU,
        dx=[dx_value, 0.02, 0.02],
        dt=3.0e-11,
        source_amplitudes=torch.zeros((1, 1, 1)),
        source_location=location,
        receiver_location=location,
        er=model,
        se=torch.zeros_like(model),
        E=electric,
        H=magnetic,
        pmlthick=0,
        fdtd_order=2,
        mode=2,
    )[1][2][0, 4, 5, 0]


def updated_ez_from_y(dy_value):
    nx, ny = 8, 10
    shape = (1, nx + 1, ny + 1, 2)
    electric = tuple(torch.zeros(shape) for _ in range(3))
    hx = torch.zeros(shape)
    hx[:] = torch.arange(ny + 1, dtype=torch.float32).reshape(1, 1, ny + 1, 1)
    magnetic = (hx, torch.zeros(shape), torch.zeros(shape))
    model = torch.full((nx, ny), 4.0)
    location = torch.tensor([[[4, 5, 0]]], dtype=torch.int32)
    return DeepGPR.compute(
        device=CPU,
        dx=[0.02, dy_value, 0.02],
        dt=3.0e-11,
        source_amplitudes=torch.zeros((1, 1, 1)),
        source_location=location,
        receiver_location=location,
        er=model,
        se=torch.zeros_like(model),
        E=electric,
        H=magnetic,
        pmlthick=0,
        fdtd_order=2,
        mode=2,
    )[1][2][0, 4, 5, 0]


def updated_ex_from_z(dz_value):
    nx, ny, nz = 6, 7, 8
    shape = (1, nx + 1, ny + 1, nz + 1)
    electric = tuple(torch.zeros(shape) for _ in range(3))
    hy = torch.zeros(shape)
    hy[:] = torch.arange(nz + 1, dtype=torch.float32).reshape(1, 1, 1, nz + 1)
    magnetic = (torch.zeros(shape), hy, torch.zeros(shape))
    model = torch.full((nx, ny, nz), 4.0)
    location = torch.tensor([[[3, 3, 4]]], dtype=torch.int32)
    return DeepGPR.compute(
        device=CPU,
        dx=[0.02, 0.02, dz_value],
        dt=2.0e-11,
        source_amplitudes=torch.zeros((1, 1, 1)),
        source_location=location,
        receiver_location=location,
        er=model,
        se=torch.zeros_like(model),
        E=electric,
        H=magnetic,
        pmlthick=0,
        source_direction=0,
        reciever_direction=0,
        fdtd_order=2,
        mode=3,
    )[1][0][0, 3, 3, 4]


directional_rows = []
for axis, update in (
    ("x", updated_ez_from_x),
    ("y", updated_ez_from_y),
    ("z", updated_ex_from_z),
):
    coarse = update(0.02)
    fine = update(0.01)
    ratio = float((fine / coarse).item())
    row = {
        "axis": axis,
        "coarse_update": float(coarse.item()),
        "fine_update": float(fine.item()),
        "fine_to_coarse_ratio": ratio,
    }
    directional_rows.append(row)
    vu.record_check(
        CHECKS,
        f"{axis}-direction field derivative scales as inverse spacing",
        float(coarse.abs()) > 0.0 and math.isclose(ratio, 2.0, rel_tol=2.0e-6),
        **row,
        expected_ratio=2.0,
    )


[PASS] x-direction field derivative scales as inverse spacing
{
  "axis": "x",
  "coarse_update": 42.35283660888672,
  "expected_ratio": 2.0,
  "fine_to_coarse_ratio": 2.0,
  "fine_update": 84.70567321777344
}
[PASS] y-direction field derivative scales as inverse spacing
{
  "axis": "y",
  "coarse_update": -42.35283660888672,
  "expected_ratio": 2.0,
  "fine_to_coarse_ratio": 2.0,
  "fine_update": -84.70567321777344
}
[PASS] z-direction field derivative scales as inverse spacing
{
  "axis": "z",
  "coarse_update": -28.235225677490234,
  "expected_ratio": 2.0,
  "fine_to_coarse_ratio": 1.9999998807907104,
  "fine_update": -56.4704475402832
}


In [6]:
nx, ny, nz = 10, 12, 14
er_pml = torch.full((nx, ny, nz), 4.0)
mr_pml = torch.ones((nx + 1, ny + 1, nz + 1))
pml_coefficients = buildpmlcoeffs(
    er_pml,
    mr_pml,
    2.0e-11,
    SPACING,
    nx,
    ny,
    nz,
    torch.tensor([2, 2, 2, 2, 2, 2], dtype=torch.int32),
    CPU,
    torch.float32,
)
x_electric, y_electric, z_electric = (
    pml_coefficients[6],
    pml_coefficients[10],
    pml_coefficients[14],
)
vu.assert_finite("axis-specific CPML coefficients", x_electric, y_electric, z_electric)
cpml_rows = {
    "x_y_relative_l2": vu.relative_l2(x_electric, y_electric),
    "y_z_relative_l2": vu.relative_l2(y_electric, z_electric),
    "x_z_relative_l2": vu.relative_l2(x_electric, z_electric),
}
vu.record_check(
    CHECKS,
    "CPML coefficients use the boundary-normal axis spacing",
    min(cpml_rows.values()) > 0.0,
    **cpml_rows,
)

def z_source_sample(dy_value):
    model = torch.full((8, 10), 4.0)
    location = torch.tensor([[[4, 5, 0]]], dtype=torch.int32)
    return DeepGPR.compute(
        device=CPU,
        dx=[0.02, dy_value, 0.01],
        dt=2.0e-11,
        source_amplitudes=torch.ones((1, 1, 1)),
        source_location=location,
        receiver_location=location,
        er=model,
        se=torch.zeros_like(model),
        pmlthick=0,
        source_direction=2,
        reciever_direction=2,
        fdtd_order=2,
        mode=2,
    )[-1][0, 0, 0]

source_coarse = z_source_sample(0.02)
source_fine = z_source_sample(0.01)
source_ratio = float((source_fine / source_coarse).item())
vu.record_check(
    CHECKS,
    "source injection uses the transverse cell dimensions",
    float(source_coarse.abs()) > 0.0
    and math.isclose(source_ratio, 2.0, rel_tol=2.0e-6),
    coarse_sample=float(source_coarse.item()),
    fine_sample=float(source_fine.item()),
    fine_to_coarse_ratio=source_ratio,
    expected_ratio=2.0,
)


[PASS] CPML coefficients use the boundary-normal axis spacing
{
  "x_y_relative_l2": 0.007846517918049198,
  "x_z_relative_l2": 0.02343195155675215,
  "y_z_relative_l2": 0.01552611922136279
}
[PASS] source injection uses the transverse cell dimensions
{
  "coarse_sample": -1411.76123046875,
  "expected_ratio": 2.0,
  "fine_sample": -2823.5224609375,
  "fine_to_coarse_ratio": 2.0
}


In [7]:
nx, ny, nt = 16, 20, 160
dt, pml = 3.0e-11, 3
source = DeepGPR.wavelet.ricker(3.0e8, nt, dt, 3.0e-9).reshape(1, nt, 1)
source_location = torch.tensor([[[6, 7, 0]]], dtype=torch.int32)
receiver_location = torch.tensor([[[6, 13, 0]]], dtype=torch.int32)
boundary_2d = vu.pml_boundary_mask((nx, ny), pml, CPU)
gradient_rows = []

def simulate_2d(er_value, se_value, order):
    return DeepGPR.compute(
        device=CPU,
        dx=SPACING,
        dt=dt,
        source_amplitudes=source,
        source_location=source_location,
        receiver_location=receiver_location,
        er=er_value,
        se=se_value,
        pmlthick=pml,
        fdtd_order=order,
        mode=2,
        model_gradient_sampling_interval=1,
        wavefield_storage_dtype=torch.float32,
    )[-1]

for order in (2, 4, 8):
    er = torch.full((nx, ny), 4.0, requires_grad=True)
    se = torch.full((nx, ny), 2.0e-4, requires_grad=True)
    receiver = simulate_2d(er, se, order)
    data_scale = receiver.detach().abs().max().clamp_min(1.0e-12)
    loss = 0.5 * (receiver / data_scale).square().sum()
    loss.backward()
    vu.assert_finite(f"2D order {order}", receiver, er.grad, se.grad)
    row = {
        "order": order,
        "receiver_absmax": float(receiver.detach().abs().max()),
        "er_gradient_absmax": float(er.grad.detach().abs().max()),
        "se_gradient_absmax": float(se.grad.detach().abs().max()),
        "er_boundary_absmax": vu.boundary_absmax(er.grad, boundary_2d),
        "se_boundary_absmax": vu.boundary_absmax(se.grad, boundary_2d),
    }
    gradient_rows.append(row)
    vu.record_check(
        CHECKS,
        f"2D unequal-spacing forward and backward remain finite at order {order}",
        row["receiver_absmax"] > 0.0
        and row["er_gradient_absmax"] > 0.0
        and row["se_gradient_absmax"] > 0.0
        and row["er_boundary_absmax"] == 0.0
        and row["se_boundary_absmax"] == 0.0,
        **row,
    )

    if order == 2:
        cell = (6, 10)

        def objective(er_value, se_value):
            value = simulate_2d(er_value, se_value, order) / data_scale
            return 0.5 * value.square().sum()

        fd_rows = []
        for parameter_name, base, gradient, step, evaluator in (
            (
                "relative permittivity",
                er.detach(),
                er.grad,
                1.0e-2,
                lambda value: objective(value, se.detach()),
            ),
            (
                "conductivity",
                se.detach(),
                se.grad,
                5.0e-5,
                lambda value: objective(er.detach(), value),
            ),
        ):
            direction = torch.zeros_like(base)
            direction[cell] = 1.0
            with torch.no_grad():
                finite_difference = float(
                    (evaluator(base + step * direction) - evaluator(base - step * direction))
                    / (2.0 * step)
                )
            adjoint = float(gradient[cell])
            relative_error = abs(adjoint - finite_difference) / max(
                abs(adjoint), abs(finite_difference), 1.0e-30
            )
            fd_row = {
                "parameter": parameter_name,
                "cell": cell,
                "step": step,
                "adjoint": adjoint,
                "finite_difference": finite_difference,
                "relative_error": relative_error,
            }
            fd_rows.append(fd_row)
            vu.record_check(
                CHECKS,
                f"unequal-spacing {parameter_name} gradient matches finite differences",
                relative_error < 2.0e-2,
                **fd_row,
                tolerance=2.0e-2,
            )


[PASS] 2D unequal-spacing forward and backward remain finite at order 2
{
  "er_boundary_absmax": 0.0,
  "er_gradient_absmax": 0.30448561906814575,
  "order": 2,
  "receiver_absmax": 428.1349792480469,
  "se_boundary_absmax": 0.0,
  "se_gradient_absmax": 8.812573432922363
}
[PASS] unequal-spacing relative permittivity gradient matches finite differences
{
  "adjoint": 0.16163843870162964,
  "cell": [
    6,
    10
  ],
  "finite_difference": 0.1617431640625,
  "parameter": "relative permittivity",
  "relative_error": 0.0006474793632075472,
  "step": 0.01,
  "tolerance": 0.02
}
[PASS] unequal-spacing conductivity gradient matches finite differences
{
  "adjoint": -5.707822322845459,
  "cell": [
    6,
    10
  ],
  "finite_difference": -5.7220458984375,
  "parameter": "conductivity",
  "relative_error": 0.00248575,
  "step": 5e-05,
  "tolerance": 0.02
}
[PASS] 2D unequal-spacing forward and backward remain finite at order 4
{
  "er_boundary_absmax": 0.0,
  "er_gradient_absmax": 0.269718

In [8]:
shape_3d = (12, 14, 16)
nt_3d, dt_3d, pml_3d = 180, 2.0e-11, 2
er_3d = torch.full(shape_3d, 4.0, requires_grad=True)
se_3d = torch.full(shape_3d, 2.0e-4, requires_grad=True)
source_3d = DeepGPR.wavelet.ricker(4.0e8, nt_3d, dt_3d, 2.0e-9).reshape(1, nt_3d, 1)
source_location_3d = torch.tensor([[[6, 5, 8]]], dtype=torch.int32)
receiver_location_3d = torch.tensor(
    [[[6, 8, 8], [6, 10, 8]]], dtype=torch.int32
)
result_3d = DeepGPR.compute(
    device=CPU,
    dx=SPACING,
    dt=dt_3d,
    source_amplitudes=source_3d,
    source_location=source_location_3d,
    receiver_location=receiver_location_3d,
    er=er_3d,
    se=se_3d,
    pmlthick=pml_3d,
    source_direction=0,
    reciever_direction=0,
    fdtd_order=2,
    mode=3,
    model_gradient_sampling_interval=1,
    wavefield_storage_dtype=torch.float32,
)
receiver_3d = result_3d[-1]
scale_3d = receiver_3d.detach().abs().max().clamp_min(1.0e-12)
(receiver_3d / scale_3d).square().mean().backward()
vu.assert_finite(
    "3D unequal-spacing fields and gradients",
    *result_3d[1],
    *result_3d[2],
    *result_3d[3],
    receiver_3d,
    er_3d.grad,
    se_3d.grad,
)
boundary_3d = vu.pml_boundary_mask(shape_3d, pml_3d, CPU)
row_3d = {
    "receiver_absmax": float(receiver_3d.detach().abs().max()),
    "er_gradient_absmax": float(er_3d.grad.detach().abs().max()),
    "se_gradient_absmax": float(se_3d.grad.detach().abs().max()),
    "er_boundary_absmax": vu.boundary_absmax(er_3d.grad, boundary_3d),
    "se_boundary_absmax": vu.boundary_absmax(se_3d.grad, boundary_3d),
}
vu.record_check(
    CHECKS,
    "3D unequal-spacing forward and backward remain finite",
    row_3d["receiver_absmax"] > 0.0
    and row_3d["er_gradient_absmax"] > 0.0
    and row_3d["se_gradient_absmax"] > 0.0
    and row_3d["er_boundary_absmax"] == 0.0
    and row_3d["se_boundary_absmax"] == 0.0,
    **row_3d,
)


[PASS] 3D unequal-spacing forward and backward remain finite
{
  "er_boundary_absmax": 0.0,
  "er_gradient_absmax": 0.019391290843486786,
  "receiver_absmax": 221.81683349609375,
  "se_boundary_absmax": 0.0,
  "se_gradient_absmax": 0.08678527921438217
}


In [9]:
cuda_row = None

def parity_case(device):
    shape = (14, 18)
    er = torch.full(shape, 4.0, device=device, requires_grad=True)
    se = torch.full(shape, 2.0e-4, device=device, requires_grad=True)
    source = DeepGPR.wavelet.ricker(3.0e8, 120, 3.0e-11, 3.0e-9).reshape(1, 120, 1).to(device)
    source_location = torch.tensor([[[6, 7, 0]]], dtype=torch.int32, device=device)
    receiver_location = torch.tensor([[[6, 11, 0]]], dtype=torch.int32, device=device)
    receiver = DeepGPR.compute(
        device=device,
        dx=SPACING,
        dt=3.0e-11,
        source_amplitudes=source,
        source_location=source_location,
        receiver_location=receiver_location,
        er=er,
        se=se,
        pmlthick=3,
        fdtd_order=2,
        mode=2,
        model_gradient_sampling_interval=1,
        wavefield_storage_dtype=torch.float32,
    )[-1]
    receiver.square().mean().backward()
    vu.assert_finite("unequal-spacing backend parity", receiver, er.grad, se.grad)
    return {
        "receiver": receiver.detach().cpu(),
        "grad_er": er.grad.detach().cpu(),
        "grad_se": se.grad.detach().cpu(),
    }

if CUDA is None:
    CUDA_METADATA = None
    vu.record_skip(
        CHECKS,
        "unequal-spacing CPU/CUDA parity",
        "CUDA is not available on this machine.",
    )
else:
    CUDA_METADATA = vu.runtime_metadata(DeepGPR, CUDA)
    cpu_parity = parity_case(CPU)
    cuda_parity = parity_case(CUDA)
    cuda_row = {
        "device": str(CUDA),
        "receiver_relative_l2": vu.relative_l2(
            cuda_parity["receiver"], cpu_parity["receiver"]
        ),
        "er_gradient_relative_l2": vu.relative_l2(
            cuda_parity["grad_er"], cpu_parity["grad_er"]
        ),
        "se_gradient_relative_l2": vu.relative_l2(
            cuda_parity["grad_se"], cpu_parity["grad_se"]
        ),
        "er_gradient_cosine": vu.cosine_similarity(
            cuda_parity["grad_er"], cpu_parity["grad_er"]
        ),
        "se_gradient_cosine": vu.cosine_similarity(
            cuda_parity["grad_se"], cpu_parity["grad_se"]
        ),
    }
    vu.record_check(
        CHECKS,
        "unequal-spacing CPU/CUDA parity",
        cuda_row["receiver_relative_l2"] < 2.0e-4
        and max(
            cuda_row["er_gradient_relative_l2"],
            cuda_row["se_gradient_relative_l2"],
        ) < 5.0e-3
        and min(
            cuda_row["er_gradient_cosine"],
            cuda_row["se_gradient_cosine"],
        ) > 0.999,
        **cuda_row,
        receiver_tolerance=2.0e-4,
        gradient_tolerance=5.0e-3,
        cosine_tolerance=0.999,
    )


[SKIPPED] unequal-spacing CPU/CUDA parity: CUDA is not available on this machine.


In [10]:
vu.save_report(
    "09_anisotropic_grid",
    CHECKS,
    METADATA,
    extra={
        "spacing": SPACING,
        "cfl_rows": cfl_rows,
        "directional_rows": directional_rows,
        "cpml_rows": cpml_rows,
        "gradient_rows": gradient_rows,
        "finite_difference_rows": fd_rows,
        "three_dimensional_row": row_3d,
        "cuda_metadata": CUDA_METADATA,
        "cuda_parity_row": cuda_row,
    },
)
print(f"Completed {len(CHECKS)} checks, including optional checks.")


Report written to /Users/llsra/Desktop/DeepGPR/tests/results/09_anisotropic_grid.json
Completed 16 checks, including optional checks.
